# 05_normalisation_mailles_ml.ipynb
Normalisation et préparation des 12 variables de la Maille pour les modèles ML (K-Means, GMM, AHP).

## 1. Chemins et Répertoires

In [2]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. Chemins et Répertoires
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent
DOSSIER_MAILLES = ROOT / "donnees_traitees" / "mailles"

FICHIER_MAILLES_GPKG = DOSSIER_MAILLES / "mailles_100m.gpkg"
FICHIER_CSV_BATI = DOSSIER_MAILLES / "densite_bati_100m.csv"
FICHIER_CSV_ROUTES = DOSSIER_MAILLES / "densite_routes_casablanca_100m.csv"
FICHIER_CSV_IMPER_RUISSEL = DOSSIER_MAILLES / "impermeable_ruissellement_100m.csv"
FICHIER_CSV_EQUIP = DOSSIER_MAILLES / "indice_equip_100m.csv"
FICHIER_CSV_SECOURS = DOSSIER_MAILLES / "distance_secours_100m.csv"
FICHIER_CSV_OUED = DOSSIER_MAILLES / "distance_oued_100m.csv"
FICHIER_CSV_MNT = DOSSIER_MAILLES / "variables_mnt_100m.csv"

FICHIER_NORMALISE_CSV = DOSSIER_MAILLES / "mailles_100m_normalisees.csv"
FICHIER_NORMALISE_PARQUET = DOSSIER_MAILLES / "mailles_100m_normalisees.parquet"

print("=" * 80)
print("   NORMALISATION DES DONNÉES DU MAILLAGE POUR LE MACHINE LEARNING")
print("=" * 80)

   NORMALISATION DES DONNÉES DU MAILLAGE POUR LE MACHINE LEARNING


## 2. Chargement et Assemblage de la Table Unique (12 Variables)

In [3]:
# -----------------------------------------------------------------------------
# 2. Chargement et Assemblage de la Table Unique
# -----------------------------------------------------------------------------
print("\n--> 1. Chargement de la grille et assemblage des variables...")
if not FICHIER_MAILLES_GPKG.exists():
    raise FileNotFoundError(f"Fichier introuvable : {FICHIER_MAILLES_GPKG}")

# Chargement de la grille de référence
df_mailles = gpd.read_file(FICHIER_MAILLES_GPKG, layer="maille")
colonnes_base = [c for c in ["id", "commune_id", "taille_m"] if c in df_mailles.columns]
df = pd.DataFrame(df_mailles[colonnes_base])

# Jointures des CSV thématiques
csv_fichiers = [
    ("Bâti", FICHIER_CSV_BATI),
    ("Routes", FICHIER_CSV_ROUTES),
    ("Imperméable & Ruissellement", FICHIER_CSV_IMPER_RUISSEL),
    ("Équipements", FICHIER_CSV_EQUIP),
    ("Secours", FICHIER_CSV_SECOURS),
    ("Distance Oued", FICHIER_CSV_OUED),
    ("Variables MNT", FICHIER_CSV_MNT),
]

for nom, chemin in csv_fichiers:
    if chemin.exists():
        print(f"  + Jointure {nom} ({chemin.name})...")
        csv_df = pd.read_csv(chemin)
        cols_a_joindre = [c for c in csv_df.columns if c == "id" or c not in df.columns]
        df = df.merge(csv_df[cols_a_joindre], on="id", how="left")
    else:
        print(f"  - {nom} absent ({chemin.name})")

# Si certaines colonnes étaient déjà dans le GPKG, on les conserve
colonnes_gpkg = [
    "pente_moy", "hand_moy", "twi_moy", "flux_max", "depression_moy",
    "densite_routes", "distance_oued", "pct_impermeable", "ruissel_moy",
    "densite_bati", "indice_equip", "distance_secours"
]
for col in colonnes_gpkg:
    if col in df_mailles.columns and col not in df.columns:
        df[col] = df_mailles[col].values

print(f"\nNombre total de mailles : {len(df):,}")
print("Colonnes disponibles   :", list(df.columns))


--> 1. Chargement de la grille et assemblage des variables...
  + Jointure Bâti (densite_bati_100m.csv)...
  + Jointure Routes (densite_routes_casablanca_100m.csv)...
  + Jointure Imperméable & Ruissellement (impermeable_ruissellement_100m.csv)...
  + Jointure Équipements (indice_equip_100m.csv)...
  + Jointure Secours (distance_secours_100m.csv)...
  + Jointure Distance Oued (distance_oued_100m.csv)...
  + Jointure Variables MNT (variables_mnt_100m.csv)...

Nombre total de mailles : 2,020,705
Colonnes disponibles   : ['id', 'commune_id', 'taille_m', 'densite_bati', 'densite_routes', 'pct_impermeable', 'ruissel_moy', 'indice_equip', 'distance_secours', 'distance_oued', 'pente_moy', 'hand_moy', 'twi_moy', 'flux_max', 'depression_moy']


## 3. Contrôle des Valeurs Manquantes (NaN) et Imputation

In [4]:
# -----------------------------------------------------------------------------
# 3. Contrôle des Valeurs Manquantes (NaN) et Imputation
# -----------------------------------------------------------------------------
print("\n--> 2. Contrôle et imputation des valeurs manquantes...")
colonnes_ml_candidates = [
    "densite_bati", "densite_routes", "pct_impermeable", "ruissel_moy",
    "distance_oued", "indice_equip", "distance_secours",
    "pente_moy", "hand_moy", "twi_moy", "flux_max", "depression_moy"
]

colonnes_presentes = [c for c in colonnes_ml_candidates if c in df.columns]
print(f"Variables ML actives ({len(colonnes_presentes)}) : {colonnes_presentes}")

for col in colonnes_presentes:
    nb_nan = df[col].isna().sum()
    if nb_nan > 0:
        valeur_impute = df[col].median()
        df[col] = df[col].fillna(valeur_impute)
        print(f"  * {col} : {nb_nan:,} NaN imputés par la médiane ({valeur_impute:.4f})")


--> 2. Contrôle et imputation des valeurs manquantes...
Variables ML actives (12) : ['densite_bati', 'densite_routes', 'pct_impermeable', 'ruissel_moy', 'distance_oued', 'indice_equip', 'distance_secours', 'pente_moy', 'hand_moy', 'twi_moy', 'flux_max', 'depression_moy']
  * densite_routes : 1,894,934 NaN imputés par la médiane (0.0000)
  * pente_moy : 22,739 NaN imputés par la médiane (1.7577)
  * hand_moy : 54,945 NaN imputés par la médiane (6.8309)
  * twi_moy : 22,739 NaN imputés par la médiane (8.0818)
  * flux_max : 22,739 NaN imputés par la médiane (21.0000)
  * depression_moy : 31,402 NaN imputés par la médiane (0.0047)


## 4. Traitement Préalable de l'Asymétrie (Log-Transform sur flux_max)

In [5]:
# -----------------------------------------------------------------------------
# 4. Traitement Préalable de l'Asymétrie (Log-Transform sur flux_max)
# -----------------------------------------------------------------------------
if "flux_max" in df.columns:
    print("\n--> 3. Transformation Logarithmique sur flux_max (réduction d'asymétrie)...")
    df["flux_max_log"] = np.log1p(df["flux_max"].clip(lower=0))
    colonnes_presentes = [c if c != "flux_max" else "flux_max_log" for c in colonnes_presentes]


--> 3. Transformation Logarithmique sur flux_max (réduction d'asymétrie)...


## 5. Normalisation Min-Max Orientée Risque Inondation [0, 1]

In [6]:
# -----------------------------------------------------------------------------
# 5. Normalisation Min-Max Orientée Risque Inondation [0, 1]
# -----------------------------------------------------------------------------
print("\n--> 4. Application de la Normalisation Min-Max orientée Risque...")

CRITERES_DIRECTS = [
    "densite_bati", "densite_routes", "pct_impermeable", "ruissel_moy",
    "indice_equip", "distance_secours", "twi_moy", "depression_moy", "flux_max_log"
]

CRITERES_INVERSES_PROTECTEURS = [
    "pente_moy",       # Pente forte = évacuation de l'eau (plat = risque fort)
    "hand_moy",        # Haut par rapport à l'oued = protégé de la crue
    "distance_oued"    # Loin du cours d'eau = protégé du débordement
]

df_norm = pd.DataFrame({"id": df["id"]})

for col in colonnes_presentes:
    c_min = df[col].min()
    c_max = df[col].max()
    
    if c_max == c_min:
        norm_val = np.zeros(len(df))
    else:
        norm_val = (df[col] - c_min) / (c_max - c_min)
    
    col_norm_nom = f"{col}"
    
    if col in CRITERES_INVERSES_PROTECTEURS:
        df_norm[col_norm_nom] = (1.0 - norm_val).clip(0.0, 1.0)
        print(f"  * {col:<18} -> INVERSE  [1 - Norm] (Min brut: {c_min:.2f}, Max brut: {c_max:.2f})")
    else:
        df_norm[col_norm_nom] = norm_val.clip(0.0, 1.0)
        print(f"  * {col:<18} -> DIRECT   [Norm]     (Min brut: {c_min:.2f}, Max brut: {c_max:.2f})")


--> 4. Application de la Normalisation Min-Max orientée Risque...
  * densite_bati       -> DIRECT   [Norm]     (Min brut: 0.00, Max brut: 1.00)
  * densite_routes     -> DIRECT   [Norm]     (Min brut: 0.00, Max brut: 0.06)
  * pct_impermeable    -> DIRECT   [Norm]     (Min brut: 0.00, Max brut: 100.00)
  * ruissel_moy        -> DIRECT   [Norm]     (Min brut: 0.20, Max brut: 1.00)
  * distance_oued      -> INVERSE  [1 - Norm] (Min brut: 0.00, Max brut: 545119.70)
  * indice_equip       -> DIRECT   [Norm]     (Min brut: 0.00, Max brut: 27.00)
  * distance_secours   -> DIRECT   [Norm]     (Min brut: 24.19, Max brut: 252798.62)
  * pente_moy          -> INVERSE  [1 - Norm] (Min brut: 0.00, Max brut: 43.84)
  * hand_moy           -> INVERSE  [1 - Norm] (Min brut: 0.00, Max brut: 327.01)
  * twi_moy            -> DIRECT   [Norm]     (Min brut: 4.28, Max brut: 18.35)
  * flux_max_log       -> DIRECT   [Norm]     (Min brut: 0.69, Max brut: 14.38)
  * depression_moy     -> DIRECT   [Norm]    

## 6. Contrôles de Validation

In [7]:
# -----------------------------------------------------------------------------
# 6. Contrôles de Validation
# -----------------------------------------------------------------------------
print("\n--> 5. Contrôles statistiques post-normalisation :")
colonnes_finales_norm = [c for c in df_norm.columns if c != "id"]
resume_stats = df_norm[colonnes_finales_norm].describe().T[["mean", "std", "min", "max"]]
print(resume_stats.round(4).to_string())

assert df_norm[colonnes_finales_norm].isna().sum().sum() == 0, "Erreur : Présence de NaN !"
assert (df_norm[colonnes_finales_norm].min().min() >= 0.0), "Erreur : Valeur < 0 détectée !"
assert (df_norm[colonnes_finales_norm].max().max() <= 1.0), "Erreur : Valeur > 1 détectée !"
print(f"\n[OK] Nombre de critères normalisés : {len(colonnes_finales_norm)}")


--> 5. Contrôles statistiques post-normalisation :
                    mean     std  min  max
densite_bati      0.0020  0.0265  0.0  1.0
densite_routes    0.0009  0.0169  0.0  1.0
pct_impermeable   0.0166  0.1158  0.0  1.0
ruissel_moy       0.1296  0.1120  0.0  1.0
distance_oued     0.8905  0.1669  0.0  1.0
indice_equip      0.0002  0.0063  0.0  1.0
distance_secours  0.3607  0.2484  0.0  1.0
pente_moy         0.9349  0.0780  0.0  1.0
hand_moy          0.9543  0.0666  0.0  1.0
twi_moy           0.2930  0.1161  0.0  1.0
flux_max_log      0.2211  0.1629  0.0  1.0
depression_moy    0.0297  0.0908  0.0  1.0

[OK] Nombre de critères normalisés : 12


## 7. Export des Données Prêtes pour le ML

In [8]:
# -----------------------------------------------------------------------------
# 7. Export des Données Prêtes pour le ML
# -----------------------------------------------------------------------------
print("\n--> 6. Sauvegarde des jeux de données normalisés...")
df_norm.to_csv(FICHIER_NORMALISE_CSV, index=False)
print(f"  [OK] CSV exporté     : {FICHIER_NORMALISE_CSV.name} ({len(df_norm):,} lignes)")

try:
    df_norm.to_parquet(FICHIER_NORMALISE_PARQUET, index=False)
    print(f"  [OK] Parquet exporté : {FICHIER_NORMALISE_PARQUET.name} (Haute performance)")
except Exception:
    pass

print("\n" + "=" * 80)
print("✅ NORMALISATION TERMINÉE AVEC SUCCÈS")
print("   La table normalisée inclut les 12 couches et est prête pour le ML.")
print("=" * 80)


--> 6. Sauvegarde des jeux de données normalisés...
  [OK] CSV exporté     : mailles_100m_normalisees.csv (2,020,705 lignes)
  [OK] Parquet exporté : mailles_100m_normalisees.parquet (Haute performance)

✅ NORMALISATION TERMINÉE AVEC SUCCÈS
   La table normalisée inclut les 12 couches et est prête pour le ML.
